# KNN y regresión logística

Detección de fraude reportado en siniestros. Ejecutar desde la raíz del repositorio con las dependencias de `requirements.txt`. Ambos modelos usan la partición estratificada 80/20 del Random Forest y se evalúan con el umbral implícito de 0,5. La codificación y el escalado de KNN se ajustan dentro de cada pliegue.

In [1]:
from time import perf_counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv('insurance_claims_limpio.csv')
X = df.drop(columns='fraud_reported')
y = df['fraud_reported']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

particion = pd.read_csv('resultados_random_forest/particion.csv')
assert set(X_train.index) == set(particion.loc[particion['split'].eq('train'), 'row_id'])
assert set(X_test.index) == set(particion.loc[particion['split'].eq('test'), 'row_id'])

categoricas = X_train.select_dtypes(exclude='number').columns.tolist()
numericas = X_train.select_dtypes(include='number').columns.tolist()

## Preprocesamiento y evaluación

La codificación conserva los marcadores `MISSING` y representa los vacíos de `authorities_contacted` mediante una categoría separada. Se mantiene el escalado de toda la matriz del notebook original. Las categorías nuevas se toleran al transformar. Las métricas usan `fraud_reported = 1` como clase positiva.

In [2]:
def crear_pipeline(modelo):
    codificador = ColumnTransformer([
        ('categoricas', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categoricas),
        ('numericas', 'passthrough', numericas),
    ])
    return Pipeline([
        ('codificar', codificador),
        ('escalar', StandardScaler()),
        ('modelo', modelo),
    ])


def evaluar(modelo, nombre):
    filas = []
    for muestra, X_eval, y_eval in [
        ('Entrenamiento', X_train, y_train),
        ('Prueba', X_test, y_test),
    ]:
        pred = modelo.predict(X_eval)
        prob = modelo.predict_proba(X_eval)[:, 1]
        filas.append({
            'Modelo': nombre,
            'Muestra': muestra,
            'AUC': roc_auc_score(y_eval, prob),
            'Precisión': precision_score(y_eval, pred, zero_division=0),
            'Recall': recall_score(y_eval, pred, zero_division=0),
            'F1': f1_score(y_eval, pred, zero_division=0),
        })
    return pd.DataFrame(filas)

## KNN

Se prueban 28 combinaciones y se elige la de mayor F1 medio en cinco pliegues estratificados. Se usa el conjunto de prueba al calcular la tabla final.

In [3]:
parametros = {
    'modelo__n_neighbors': [3, 5, 7, 9, 11, 15, 21],
    'modelo__weights': ['uniform', 'distance'],
    'modelo__metric': ['euclidean', 'manhattan'],
}
busqueda = GridSearchCV(
    crear_pipeline(KNeighborsClassifier()),
    parametros,
    scoring='f1',
    cv=StratifiedKFold(n_splits=5),
    n_jobs=1,
    error_score='raise',
)
inicio = perf_counter()
busqueda.fit(X_train, y_train)
tiempo_knn = perf_counter() - inicio
print('Mejores hiperparámetros:', busqueda.best_params_)
print(f'F1 medio de validación: {busqueda.best_score_:.4f}')
print(f'Tiempo de búsqueda y ajuste final: {tiempo_knn:.2f} s')
resultados_knn = evaluar(busqueda.best_estimator_, 'KNN')
print(resultados_knn.round(4).to_string(index=False))

Mejores hiperparámetros: {'modelo__metric': 'euclidean', 'modelo__n_neighbors': 3, 'modelo__weights': 'uniform'}
F1 medio de validación: 0.2571
Tiempo de búsqueda y ajuste final: 5.32 s
Modelo       Muestra    AUC  Precisión  Recall     F1
   KNN Entrenamiento 0.8630     0.7405  0.4899 0.5897
   KNN        Prueba 0.5003     0.2667  0.1633 0.2025


## Regresión logística de referencia

Los pesos de clase son inversamente proporcionales a sus frecuencias en entrenamiento. Se usan los hiperparámetros de la referencia, sin búsqueda adicional.

In [4]:
logistica = crear_pipeline(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
inicio = perf_counter()
logistica.fit(X_train, y_train)
tiempo_logistica = perf_counter() - inicio
print(f'Tiempo de ajuste: {tiempo_logistica:.2f} s')
resultados_logistica = evaluar(logistica, 'Regresión logística')
print(resultados_logistica.round(4).to_string(index=False))

Tiempo de ajuste: 0.04 s
             Modelo       Muestra    AUC  Precisión  Recall     F1
Regresión logística Entrenamiento 0.9641     0.7480  0.9444 0.8348
Regresión logística        Prueba 0.7806     0.5439  0.6327 0.5849


## Figura para el informe

La sensibilidad al número de vecinos se mide en validación. La curva ROC de prueba corresponde al modelo elegido.

In [5]:
resultados_cv = pd.DataFrame(busqueda.cv_results_)
f1_por_k = resultados_cv.groupby('param_modelo__n_neighbors')['mean_test_score'].max()
fpr, tpr, _ = roc_curve(y_test, busqueda.predict_proba(X_test)[:, 1])

fig, (sensibilidad, roc) = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
sensibilidad.plot(f1_por_k.index, f1_por_k, marker='o')
sensibilidad.set(xlabel='Número de vecinos', ylabel='Mejor F1 medio', title='Sensibilidad en validación')
roc.plot(fpr, tpr, label=f"AUC = {resultados_knn.iloc[1]['AUC']:.3f}")
roc.plot([0, 1], [0, 1], '--', color='gray', label='Azar')
roc.set(xlabel='Tasa de falsos positivos', ylabel='Recall', title='Curva ROC en prueba')
roc.legend(loc='lower right')
carpeta_figuras = Path('figuras_informe')
carpeta_figuras.mkdir(exist_ok=True)
fig.savefig(carpeta_figuras / 'knn_diagnostico.png', dpi=180)
plt.close(fig)